##### Webscraping Program Curriculums
> Two Methods:
- **beautifulsoup (Python)**
- **selenium (JavaScript)**

In [2]:
# Beautifulsoup import.
from bs4 import BeautifulSoup
from urllib.request import urlopen
from requests_html import HTMLSession

from urllib.error import HTTPError
import csv

from attr import attrs


##### Import Statments 
* *pip install beautifulsoup4*
* *pip install requests* 
* *pip install requests-html*
* *pip install lxml_html_clean*
* *pip install csv*
* *pip install attrs*

In [3]:
import requests

url = "https://catalog.ncsu.edu/undergraduate/management/accounting/accounting-bs/#planrequirementstext"
response = requests.get(url)
response.raise_for_status()

bs = BeautifulSoup(response.text, "html.parser")

In [ ]:
# Creating the bs object. 
# try:
    # html = urlopen('https://catalog.ncsu.edu/undergraduate/agriculture-life-sciences/food-bioprocessing-nutrition-science/nutrition-sciences-bs-applied-nutrition-concentration/#semestersequencetext')
# except HTTPError as e:
    # print(e)
# except URLError as e:
    # print('The server could not be found!')
# else:
    # print('It Worked!')
# consider swapping the 'html.parser' for 'lxml' if you have it installed. It is faster.
# bs = BeautifulSoup(html, 'html.parser')

#### **Why am I getting info from the wrong?**

* * the code is not fetching the anchor (#semestersequencetext) because URL fragments are never sent to the server. *SO:* the server returns the base page, not the section you expected. 
* * when you make the URL request, everything but the # is a fragment identifier.  
* * The browser use fragments client-side only to scroll to the section of the page. 
* * Fragments are **NOT** sent in HTTP requests.  S0, the server will only receive the URL without the *#*.
* * **THEREFORE** *urlopen()* cannot fetch that subsection - it fetches the **whole page**. 
> * * Why the HTML looks different from what you see in the browser?
>> NCSU's catalog site is built with JavaScript that loads content dynamically:
>> When you fetch the page with Python:
>> * You get the **raw HTML**, befor JavaScript runs. 
>> * The "Semester Sequence" section is inserted by JavaScript ater page load. 
>> * urlopen() does not execute JavaScript. 
>> * So your **bs object** contains only the static HTML. 

> ##### TO VERIFY:
>> * * Open the page in the browser -> right-click -> "View Page Source". 
>> notice that the *Semester Sequence* is **not** contained in the raw HTML.
>> *It appears only in the rendered DOM, after JavaScript runs.*

##### Alternate Method to *CORRECTLY SCRAPE*

> * USE *requests_html* 

> *  Use Selenium:
>>from selenium import webdriver
>>driver = webdriver.Chrome()
>>driver.get('https://catalog.ncsu.edu/.../#semestersequencetext')
>>html = driver.page_source

> * Find API endpoint the site uses 
> *NCSU's catalog often loads sections from JSON endpoints.  You can inspect the Network tab in DevTools to find the real datasource*




In [9]:
if False:
    rows = bs.tbody.find_all("tr", class_=["even", "odd"])

    for row in rows:
        for cell in row.find_all("td"):
            match cell["class_"]:
                case "titlecol":
                    if cell:
                        cell1 = cell.get_text(strip=True)
                    else:
                        cell1 = "none"
                case "coursecol":
                    if cell:
                        cell2 = cell.get_text(strip=True)
                    else:
                        cell2 = "none"
                case "creditcol":
                    if cell:
                        cell3 = cell.get_text(strip=True)
                    else:
                        cell3 = "none"
                case _:
                    continue
        print(f'{cell1}, {cell2}, {cell3}')
print('ignore Attempt1')
            


ignore Attempt1


In [10]:
# Match attempt 2
if False:
    print(bs.tbody.tr.td.text)
    # print(bs.tbody.tr.get("class", ["even", "odd"])[0])
    print("\n")
    rows = bs.tbody.find_all("tr", class_=["even", "odd"])
    for row in rows:
        # print(row)
        i = 0 
        title = "error1"
        course = "error2"
        credits = "error3"
        for cell in row.find_all("td"): # This right here is creating a list.  
            # print(cell)
            match cell.get("class", []):
                case c if "titlecol" in c:
                    i = 1
                    title = cell.get_text(strip=True)
                    if title == "":
                        title = "Blank Line"
                case b if cell == row.find_all("td")[1]:
                # case c if "coursecol" in c:
                    i = 2
                    course = cell.get_text(strip=True)
                    if course == "":
                        course = "Blank Line"
                case c if "hourscol" in c:
                    i = 3
                    try:
                        credits = cell.get_text(strip=True)
                        if credits == "":
                            credits = "0"
                    except:
                        credits = "_"
                    
                case _:
                    match i: 
                        case 1:
                            course = "None"
                        case 2:
                            credits = "None"
                        case 3:
                            title = "None"
                        case 0:
                            title = bs.tbody.tr.td.text
                            course = ""
                            credits = ""
        print(f'{title}, {course}, {credits}')
print('Ignore Attempt2')

Ignore Attempt2


In [67]:
# Match attempt 3
# print(bs.tbody.tr.get("class", ["even", "odd"])[0])
if False:
    print("\n")
    rows = bs.tbody.find_all("tr", class_=["even", "odd"])
    for row in rows:
        # print(row)
        i = 0 
        title = "error1"
        course = "error2"
        credits = "error3"
        # if len(row.find_all("td")) < 3:
                # row.append("<td></td>")
        # The problem here is that sometimes the list isn't long enough. 
        for index, cell in enumerate(row.find_all("td")): # If I created a cells list, I could do "range(len(cells))"
            match index:
                case 0:
                    i = 1
                    title = cell.get_text(strip=True)
                    if title == "":
                        title = "Blank Line"
                case 1:
                    i = 2
                    course = cell.get_text(strip=True)
                    if course == "" and title == "Blank Line":
                        course = "Blank Line"
                    elif course == "":
                        course = "_"
                case 2:
                    i = 3
                    credits = cell.get_text(strip=True)
                    if credits == "" and title == "Blank Line":
                        credits = "Blank Line"
                    elif credits == "" and course == "_":
                        credits = "_"
                    else:
                        credits = "0"
                case _:
                    match i: 
                        case 1:
                            course = "ErrorA"
                        case 2:
                            credits = "ErrorB"
                        case 3:
                            title = "ErrorC"
                        case 0:
                            title = bs.tbody.tr.td.text
                            course = ""
                            credits = "_"
        print(f'[{title}], [{course}], [{credits}]')
print(f'Ignore Attempt3')

Ignore Attempt3


In [4]:
# Match attempt 4
# I got this one to work good enough.
# Let me see how many pages it'll work for. 

rows = bs.tbody.find_all("tr", class_=["even", "odd"])
for row in rows:
    # print(row)
    i = 0 
    title = "error1"
    course = "error2"
    # credits = "error3"

    for index, cell in enumerate(row.find_all("td")): # I could do "range(len(cells))"
        match index:
            case 0:
                i = 1
                title = cell.get_text(strip=True)
                if title == "":
                    title = "Blank Line"
            case 1:
                i = 2
                course = cell.get_text(strip=True)
                # credits = "0" # Here is a working fix.
                if course == "" and title == "Blank Line":
                    course = "Blank Line"
                elif course == "":
                    course = "_"
                    credits = "_"
            case 2:
                i = 3
                credits = cell.get_text(strip=True)
                # if credits == "" and title == "Blank Line":
                    # credits = "Blank Line"
                # elif credits == "" and course == "_":
                    # credits = "_"
                if credits == "":
                    credits = "0"
            case _:
                match i: 
                    case 1:
                        course = "ErrorA"
                    case 2:
                        credits = "ErrorB"
                    case 3:
                        title = "ErrorC"
                    case 0: # Because the 0 case is clearly triggered.  Values should all be loaded. so credits should be over written.  
                        title = bs.tbody.tr.td.text
                        credits = "_"
                        course = "_"
                        
    print(f'[{title}], [{course}], [{credits}]')


[Humanities and Social Sciences], [_], [_]
[Acad Writing Research1], [4], [_]
[Select one of the following:], [3], [_]
[COM 110], [Public Speaking], [0]
[COM 112], [Interpersonal Communication], [0]
[COM 211], [Argumentation and Advocacy], [0]
[PSY 200], [Introduction to Psychology], [3]
[Select one of the following:], [3], [3]
[ARE 201], [Introduction to Agricultural & Resource Economics], [0]
[ARE 201A], [Introduction to Agricultural & Resource Economics], [0]
[EC 201], [Principles of Microeconomics], [0]
[EC 202], [Principles of Macroeconomics], [3]
[Select one of the following:], [3], [3]
[ENG 331], [Communication for Engineering and Technology], [0]
[ENG 332], [Communication for Business and Management], [0]
[ENG 333], [Communication for Science and Research], [0]
[GEP Humanities], [6], [0]
[GEP Elective], [3], [0]
[Select one of the following: (verify requirement)], [_], [_]
[MIE 306], [Managing Ethics in Organizations], [0]
[PHI 214], [Issues in Business Ethics], [0]
[PHI 221], 

# Print the file to PDF and use a PDF -> CSV reader -> xml. (fastest method)

##### Here I'm going to attempt to add my code to this CSV Section

In [6]:
# Creating a CSV File of the Table. 
csvFile = open('Accounting.csv', 'wt+')
writer = csv.writer(csvFile)
try: 
    for row in rows:
        csvRow = []
        for cell in row.findAll('td', class_=["codecol","titlecol","hourscol"]):
            csvRow.append(cell.get_text())
        writer.writerow(csvRow)
finally:
    csvFile.close()

/tmp/ipykernel_3765/117111424.py:7: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  for cell in row.findAll('td', class_=["codecol","titlecol","hourscol"]):


In [7]:
# This part should give me a CSV file called "nutrition_Science.csv"
csvFile = open('accounting_NCSU.csv', 'wt+')
writer = csv.writer(csvFile)
rows = bs.tbody.find_all("tr", class_=["even", "odd"])

for row in rows:
    csvRow = []

    i = 0 
    title = "error1"
    course = "error2"
    # credits = "error3"

    for index, cell in enumerate(row.find_all("td")): # I could do "range(len(cells))"
        match index:
            case 0:
                i = 1
                title = cell.get_text(strip=True)
                if title == "":
                    title = "Blank Line"
            case 1:
                i = 2
                course = cell.get_text(strip=True)
                # credits = "0" # Here is a working fix.
                if course == "" and title == "Blank Line":
                    course = "Blank Line"
                elif course == "":
                    course = "_"
                    credits = "_"
            case 2:
                i = 3
                credits = cell.get_text(strip=True)
                # if credits == "" and title == "Blank Line":
                    # credits = "Blank Line"
                # elif credits == "" and course == "_":
                    # credits = "_"
                if credits == "":
                    credits = "0"
            case _:
                match i: 
                    case 1:
                        course = "ErrorA"
                    case 2:
                        credits = "ErrorB"
                    case 3:
                        title = "ErrorC"
                    case 0: # Because the 0 case is clearly triggered.  Values should all be loaded. so credits should be over written.  
                        title = bs.tbody.tr.td.text
                        credits = "_"
                        course = "_"
                        
    csvRow.append([title, course, credits])
    writer.writerow(csvRow)

csvFile.close()

##### In the Next Section I will use JavaScript
> pip install webdriver-manager

## This is the Beginning of the Selenium Section


In [1]:
# Working with Selenium JavaScript
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

##### Selenium 
> - Selenium is a web testing library that can be used to automate browser interactions.
> - Selenium can be used to scrape data from websites that use JavaScript to load content. 

* would be interesting to enumerate the ID#. 

In [3]:
# Attempt 4

blocks = driver.find_elements(By.CSS_SELECTOR,"#block_acf-block-6a1d9b1ea64fe-acc-content-panel > div > ul.wp-block-list")
for index, item in enumerate(blocks):
    html = item.get_attribute("innerHTML")
    parts = html.split(">, ")
    for part in parts:
        print(part)
    # Find the header ABOVE the <ul>
    # headers1 = item.find_elements(By.XPATH, "//h3")
    # headers1 = item.find_elements(By.XPATH, "//h3[contains(@class,'wp-block-heading')]")
    # headers1 = item.find_elements(By.XPATH, "./preceding-sibling::div[contains(@class,'acf-innerblocks-container')][1]//*[contains(@class,'wp-block-heading')]")
    # headers1 = item.find_element(By.XPATH, "./preceding-sibling::div[contains(@class,'accordion-header')][1]//*[contains(@class,'wp-block-heading')]").text.strip()        

    # print(f"\n--- {headers1}: {index+1} ---")

    # Extract list items
    # items = item.find_elements(By.TAG_NAME, "li")
    # for li in items:
        # print(" -", li.text.strip())



<li>FYS 101, First Year Seminar (3)</li>



<li>CH 105, General Chemistry (with lab) or CH 107 if qualified&nbsp;(4)</li>



<li>BI 105, Introductory&nbsp;to Cell Biology (3) or MA 106, Calculus &amp; Geometry&nbsp;(4) or equivalent (MA 104/105)</li>



<li>Core (TI, PCA, or SW)*&nbsp;or Liberal Education Elective (3)</li>



<li>PX 100, Exploring Pharmacy 1 (1)</li>


<li>FYS 102, First Year Seminar (3)</li>



<li>CH 106, General Chemistry (with lab) (4)</li>



<li>BI 105, Introductory&nbsp;to Cell Biology (3) or MA 106, Calculus &amp; Geometry&nbsp;(4) or equivalent (MA 104/105)</li>



<li>Core (TI, PCA, or SWA)* (3)</li>



<li>PWB, Physical Well Being (1)</li>



<li>PX 102, Exploring Pharmacy 2 (1)</li>


<li>GHS 201-2019,&nbsp;Global and Historical Studies (3)</li>



<li>CH 351, Organic Chemistry with lab (4)</li>



<li>PX 326, Human Anatomy &amp; Physiology 1 (4)</li>



<li>PX 201, The Science of Pharmacy (1)</li>



<li>Core (PCA, TI, or SW)* (3)</li>


<li>GHS 201-209, 

In [ ]:
blocks = driver.find_elements(By.CSS_SELECTOR,"#block_acf-block-6a1d9b1ea64fe-acc-content-panel > div > ul.wp-block-list")
for index, item in enumerate(blocks):
    headers1 = item.find_elements(By.XPATH, "//h3")
    # headers2 = item.find_elements(By.XPATH, "./preceding-sibling::div[contains(@class,'acf-innerblocks-container')][1]//*[contains(@class,'wp-block-heading')]"")
    # print(f"\n--- {headers1}: {index+1} ---")

In [2]:
# Selenium Attempt 3
# Import Statements
from selenium.webdriver.common.by import By
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

# Options Object
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

# Create Chrome Driver Object
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options,
)

# Page Fetching
driver.get("https://www.butler.edu/health-professions/doctor-of-pharmacy/curriculum/")
time.sleep(3)  # allow JS to load

In [2]:
# Select the accordion content blocks

# blocks = driver.find_elements(By.CSS_SELECTOR, "div.acf-innerblocks-container")
blocks = driver.find_elements(By.CSS_SELECTOR, "h3.wp-block-heading")
# blocks = driver.find_elements(By.CSS_SELECTOR, "#post-41 > div > section.wysiwyg.wysiwyg-primary-block.alignfull > div > div > div > div")
# blocks = driver.find_elements(By.TAG_NAME, "li") #INTERESTINGLY GOT OUTPUT WRONG BLOCKS.
for index, item in enumerate(blocks):
    headers = item.find_element(By.CSS_SELECTOR, "h3.wp-block-heading")
    # items = item.find_elements(By.TAG_NAME, "li")
    print(f"\n--- {headers.text.strip()}: {index+1} ---")
    # for li in items:
        # print(" -", li.text.strip())    

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"h3.wp-block-heading"}
  (Session info: chrome=149.0.7827.155); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
#0 0x63255e1c93da <unknown>
#1 0x63255dbab139 <unknown>
#2 0x63255dbff884 <unknown>
#3 0x63255dbffad1 <unknown>
#4 0x63255dbf47c6 <unknown>
#5 0x63255dbf46a7 <unknown>
#6 0x63255dc47b5c <unknown>
#7 0x63255dbf2e4f <unknown>
#8 0x63255dbf3c31 <unknown>
#9 0x63255e18f9f7 <unknown>
#10 0x63255e18e203 <unknown>
#11 0x63255e178f16 <unknown>
#12 0x63255e18ed9a <unknown>
#13 0x63255e160eb0 <unknown>
#14 0x63255e1b5d58 <unknown>
#15 0x63255e1b5ef5 <unknown>
#16 0x63255e1c7f5e <unknown>
#17 0x7132b826aaa4 <unknown>
#18 0x7132b82f7c6c <unknown>


In [ ]:
# Clean Code Block

# finds all elements on the page that are h3 with the appropriate tag.id
blocks = driver.find_elements(By.CSS_SELECTOR, "h3.wp-block-heading")
# basically makes the list like a dictionary
for index, item in enumerate(blocks):
    # This should capture the same [first] header.
    html = item.get_attribute("innerHTML")
    # headers = driver.find_element(By.CSS_SELECTOR, "h3.wp-block-heading")
    print(f"\n--- {headers.text}: {index+1} ---")
    print(item.text)
    print(headers.getText())
    # print(f"\n--- {item.text.strip()}: {index} ---")

# Interpretation: 
# It appears to be finding h3 tage with the correct class. 
# however the headers variable that is created is unable to be processed with the text.strip. 


--- : 1 ---



AttributeError: 'WebElement' object has no attribute 'getText'

In [ ]:
# Correct Java Formatted Code 

List<WebElement> blocks = driver.findElements(
    By.cssSelector("h3.wp-block-heading")
);

for (int i = 0; i < blocks.size(); i++) {
    WebElement item = blocks.get(i);

    System.out.println(
        "\n--- " + item.getText().trim() + ": " + (i + 1) + " ---"
    );
}

SyntaxError: cannot assign to comparison (2429463246.py, line 3)

In [ ]:
# Take a Break
driver.quit()

In [ ]:
# Selenium Attempt 2

# Import Statements
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import time

# Options Object
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

# Chrome Driver Object
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options,
)

# Page Fetching
driver.get("https://www.butler.edu/health-professions/doctor-of-pharmacy/curriculum/")
time.sleep(3)  # allow JS to load

# ⭐ THIS is the correct Selenium selector
blocks = driver.find_all(By.CSS_SELECTOR, "#block_acf-block-6a1d9b1ea651c-acc-content-panel > div")
for index, item in enumerate(blocks):
    item_List = item.find_all(By.TAG_NAME, "h3" OR "li")
    for sub_item in item_List:
        print(sub_item.text.strip())


# print(f"Found {len(blocks)} blocks")

# for b in blocks:
    # print(b.text.strip())
    # print("-----")

# Close Chrome Driver
driver.quit()


Found 1 blocks

-----


In [3]:
# Create a new instance of the Chrome driver
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options,
 )
driver.get('https://www.butler.edu/health-professions/doctor-of-pharmacy/curriculum/')
time.sleep(2)
driver.quit()

In [4]:
row_counter=0
print([title, course, credits])
# Idea: I need to specify which attributes I want to print.  
# Example -> Attempt to print a table data or table body element with a sepcific class. 

csvRow.append([title, course, credits])
writer.writerow(csvRow)
row_counter += 1

NameError: name 'title' is not defined

In [19]:
print(driver.title)
# print(driver.current_url)

MaxRetryError: HTTPConnectionPool(host='localhost', port=38921): Max retries exceeded with url: /session/ad2d4466ce713eb72640c2f9b8cd3fc0/title (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x78e29de31790>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [ ]:

# Go to the desired webpage
driver.get('https://catalog.coastal.edu/preview_program.php?catoid=33&poid=5936&returnto=1411')

# Extract the page source after JavaScript has rendered
html = driver.page_source



In [ ]:
# This is Revised code from earlier for scraping the "correct" target page:
from selenium import webdriver
driver = webdriver.Chrome()
driver.get('https://catalog.ncsu.edu/.../#semestersequencetext')
html = driver.page_source
# I believe the problem with that the # was not read as the anchor for the bs python.  So the page selected is prioritized incorrectly. 


##### In the next section I will use a pandas web reader.

In [ ]:
# Alternate method using pandas.
# Looks like there are a couple more steps with this one. 
import pandas as pd

url = "https://catalog.ncsu.edu/undergraduate/agriculture-life-sciences/food-bioprocessing-nutrition-science/nutrition-sciences-bs-applied-nutrition-concentration/#semestersequencetext"

tables = pd.read_html(url)
len(tables)


tables[0].to_csv("nutrition_sciences(2).csv", index=False)
